# Dynamic Proramming

In [11]:
import numpy as np

UP = 0
RIGHT = 1
DOWN = 2
LEFT = 3

ACTIONS = {
    UP: (-1, 0),
    RIGHT: (0, 1),
    DOWN: (1, 0),
    LEFT: (0, -1)
}

BACKWARD = {
    UP: DOWN,
    DOWN: UP,
    RIGHT: LEFT,
    LEFT: RIGHT
}

ARROWS = {
    UP: "↑",
    RIGHT: "→",
    DOWN: "↓",
    LEFT: "←"
}


class GridWorld:
    def __init__(self):
        self.n = 10 # n by n grid world
        self.terminal = (9, 9)

        self.p_forward = 0.7
        self.p_backward = 0.3

        self.wall_reward = -3
        self.step_reward = -1
        self.goal_reward = 0

        self.P = self.make_transition_model()

    def is_terminal(self, state):
        return state == self.terminal

    def inside_grid(self, state):
        x, y = state
        return 0 <= x < self.n and 0 <= y < self.n

    def move(self, state, action):
        if self.is_terminal(state):
            return state, 0, True

        x, y = state
        dx, dy = ACTIONS[action]
        next_state = (x + dx, y + dy)

        if not self.inside_grid(next_state):
            return state, self.wall_reward, False

        if self.is_terminal(next_state):
            return next_state, self.goal_reward, True

        return next_state, self.step_reward, False

    def transitions(self, state, action):
        backward_action = BACKWARD[action]

        next1, reward1, done1 = self.move(state, action)
        next2, reward2, done2 = self.move(state, backward_action)

        return [
            (self.p_forward, next1, reward1, done1),
            (self.p_backward, next2, reward2, done2)
        ]

    def make_transition_model(self):
        P = {}

        for x in range(self.n):
            for y in range(self.n):
                state = (x, y)
                P[state] = {}

                for action in ACTIONS:
                    P[state][action] = self.transitions(state, action)

        return P

    def step(self, state, action):
        if np.random.rand() < self.p_forward:
            real_action = action
        else:
            real_action = BACKWARD[action]

        return self.move(state, real_action)

In [9]:
class Agent:
    def __init__(self, env, gamma=0.9, theta=0.0001):
        self.env = env
        self.gamma = gamma
        self.theta = theta

        self.x = 0
        self.y = 0

        self.P = env.P

        # random value function
        self.V = np.random.rand(env.n, env.n)
        self.V[9, 9] = 0

        # random policy
        self.policy = np.zeros((env.n, env.n), dtype=int)
        self.make_random_policy()

    def make_random_policy(self):
        for x in range(self.env.n):
            for y in range(self.env.n):

                if (x, y) == self.env.terminal:
                    self.policy[x, y] = -1

                else:
                    self.policy[x, y] = np.random.randint(0, 4)

    def q_value(self, state, action):
        value = 0

        for prob, next_state, reward, done in self.P[state][action]:
            nx, ny = next_state
            value += prob * (reward + self.gamma * self.V[nx, ny])

        return value

    def policy_iteration(self):
        while True:

            # policy evaluation
            while True:
                delta = 0

                for x in range(self.env.n):
                    for y in range(self.env.n):

                        state = (x, y)

                        if self.env.is_terminal(state):
                            continue

                        old_value = self.V[x, y]
                        action = self.policy[x, y]

                        self.V[x, y] = self.q_value(state, action)

                        delta = max(delta, abs(old_value - self.V[x, y]))

                if delta < self.theta:
                    break

            # policy improvement
            policy_stable = True

            for x in range(self.env.n):
                for y in range(self.env.n):

                    state = (x, y)

                    if self.env.is_terminal(state):
                        continue

                    old_action = self.policy[x, y]

                    q_values = []

                    for action in ACTIONS:
                        q_values.append(self.q_value(state, action))

                    best_action = np.argmax(q_values)
                    self.policy[x, y] = best_action

                    if old_action != best_action:
                        policy_stable = False

            if policy_stable:
                break

    def value_iteration(self):
        while True:
            delta = 0

            for x in range(self.env.n):
                for y in range(self.env.n):

                    state = (x, y)

                    if self.env.is_terminal(state):
                        continue

                    old_value = self.V[x, y]

                    q_values = []

                    for action in ACTIONS:
                        q_values.append(self.q_value(state, action))

                    self.V[x, y] = max(q_values)

                    delta = max(delta, abs(old_value - self.V[x, y]))

            if delta < self.theta:
                break

        # extract policy after value iteration
        for x in range(self.env.n):
            for y in range(self.env.n):

                state = (x, y)

                if self.env.is_terminal(state):
                    continue

                q_values = []

                for action in ACTIONS:
                    q_values.append(self.q_value(state, action))

                self.policy[x, y] = np.argmax(q_values)

    def random_start(self):
        while True:
            self.x = np.random.randint(0, self.env.n)
            self.y = np.random.randint(0, self.env.n)

            if not self.env.is_terminal((self.x, self.y)):
                break

    def step(self):
        state = (self.x, self.y)

        if self.env.is_terminal(state):
            return state, 0, True

        action = self.policy[self.x, self.y]

        next_state, reward, done = self.env.step(state, action)

        self.x, self.y = next_state

        return next_state, reward, done

    def episode(self, max_steps=1000):
        self.random_start()

        total_reward = 0
        path = []

        for i in range(max_steps):
            state = (self.x, self.y)
            next_state, reward, done = self.step()

            path.append((state, next_state, reward))
            total_reward += reward

            if done:
                break

        return path, total_reward

    def print_policy(self):
        for x in range(self.env.n):
            row = ""

            for y in range(self.env.n):
                if (x, y) == self.env.terminal:
                    row += " T "
                else:
                    action = self.policy[x, y]
                    row += " " + ARROWS[action] + " "

            print(row)

    def print_values(self):
        print(np.round(self.V, 2))

In [10]:
env = GridWorld()
agent = Agent(env, gamma=0.9)

agent.value_iteration()
agent.print_policy()
agent.print_values()

 ↓  →  →  →  →  →  →  →  →  ↓ 
 ↓  →  →  →  →  →  →  →  →  ↓ 
 ↓  ↓  →  →  →  →  →  →  →  ↓ 
 ↓  ↓  ↓  →  →  →  →  →  →  ↓ 
 ↓  ↓  ↓  ↓  →  →  →  →  →  ↓ 
 ↓  ↓  ↓  ↓  ↓  →  →  →  →  ↓ 
 ↓  ↓  ↓  ↓  ↓  ↓  →  →  →  ↓ 
 ↓  ↓  ↓  ↓  ↓  ↓  ↓  →  →  ↓ 
 ↓  ↓  ↓  ↓  ↓  ↓  ↓  ↓  →  ↓ 
 →  →  →  →  →  →  →  →  →  T 
[[-11.08 -10.3  -10.01  -9.89  -9.82  -9.77  -9.71  -9.63  -9.54  -9.43]
 [-10.3   -9.9   -9.71  -9.58  -9.46  -9.33  -9.16  -8.96  -8.7   -8.39]
 [-10.01  -9.71  -9.53  -9.38  -9.22  -9.03  -8.79  -8.5   -8.14  -7.69]
 [ -9.89  -9.58  -9.38  -9.2   -9.    -8.75  -8.45  -8.07  -7.6   -7.02]
 [ -9.82  -9.46  -9.22  -9.    -8.74  -8.43  -8.05  -7.58  -6.99  -6.26]
 [ -9.77  -9.33  -9.03  -8.75  -8.43  -8.05  -7.57  -6.98  -6.25  -5.34]
 [ -9.71  -9.16  -8.79  -8.45  -8.05  -7.57  -6.98  -6.25  -5.34  -4.21]
 [ -9.63  -8.96  -8.5   -8.07  -7.58  -6.98  -6.25  -5.34  -4.21  -2.8 ]
 [ -9.54  -8.7   -8.14  -7.6   -6.99  -6.25  -5.34  -4.21  -2.8   -1.06]
 [ -9.43  -8.39  -7.69  -7.02  -6